 ## Fine-tuning DistilBERT on IMDB
 HuggingFace Trainer API
**Task:** Binary Sentiment Classification
**Model:** DistilBERT (smaller, faster BERT)
**Dataset:** IMDB Movie Reviews (50,000 reviews)

In [1]:
! pip  install transformers datasets evaluate accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.4 MB/s eta 0:00:00


In [2]:

# Imports
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import evaluate

print("All imports successful!")
print(f"PyTorch version      : {torch.__version__}")
print(f"GPU available        : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name             : {torch.cuda.get_device_name(0)}")


All imports successful!
PyTorch version      : 2.10.0+cu128
GPU available        : True
GPU name             : Tesla T4


In [4]:
# ── Load IMDB Dataset ─────────────────────────
print("Loading IMDB dataset...")
dataset = load_dataset("imdb")

print(f"\nDataset structure:")
print(dataset)

print(f"\nFirst training example:")
print(f"Text  : {dataset['train'][0]['text'][:200]}...")
print(f"Label : {dataset['train'][0]['label']}")
print(f"(0 = Negative, 1 = Positive)")

print(f"\nDataset sizes:")
print(f"Train : {len(dataset['train']):,}")
print(f"Test  : {len(dataset['test']):,}")

# Use smaller subset for faster training
# Full dataset takes ~30 mins on free Colab
TRAIN_SIZE = 3000
TEST_SIZE  = 500

small_train = dataset['train'].shuffle(seed=42)\
                               .select(range(TRAIN_SIZE))
small_test  = dataset['test'].shuffle(seed=42)\
                              .select(range(TEST_SIZE))

print(f"\nUsing subset:")
print(f"Train : {len(small_train):,}")
print(f"Test  : {len(small_test):,}")

# Check class balance
labels      = small_train['label']
n_positive  = sum(labels)
n_negative  = len(labels) - n_positive
print(f"\nClass balance:")
print(f"Positive: {n_positive} ({n_positive/len(labels)*100:.1f}%)")
print(f"Negative: {n_negative} ({n_negative/len(labels)*100:.1f}%)")

Loading IMDB dataset...


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]


Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

First training example:
Text  : I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ev...
Label : 0
(0 = Negative, 1 = Positive)

Dataset sizes:
Train : 25,000
Test  : 25,000

Using subset:
Train : 3,000
Test  : 500

Class balance:
Positive: 1489 (49.6%)
Negative: 1511 (50.4%)


In [5]:
# ── Tokenize Dataset ──────────────────────────
MODEL_NAME = "distilbert-base-uncased"

print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    """
    Tokenize a batch of examples.
    truncation=True  → cut if longer than max_length
    max_length=256   → DistilBERT max is 512
                       256 is enough for most reviews
                       and faster to train!
    """
    return tokenizer(
        examples['text'],
        truncation = True,
        max_length = 256
    )

print("Tokenizing dataset...")
tokenized_train = small_train.map(
    tokenize_function,
    batched      = True,  # process in batches (faster!)
    remove_columns = ['text']  # remove text column
)
tokenized_test = small_test.map(
    tokenize_function,
    batched      = True,
    remove_columns = ['text']
)

print(f"\nTokenized train features: {tokenized_train.features}")
print(f"\nFirst tokenized example:")
print(f"input_ids length : {len(tokenized_train[0]['input_ids'])}")
print(f"label            : {tokenized_train[0]['label']}")
print(f"First 10 ids     : {tokenized_train[0]['input_ids'][:10]}")

Loading tokenizer: distilbert-base-uncased


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing dataset...


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]


Tokenized train features: {'label': ClassLabel(names=['neg', 'pos']), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8'))}

First tokenized example:
input_ids length : 179
label            : 1
First 10 ids     : [101, 2045, 2003, 2053, 7189, 2012, 2035, 2090, 3481, 3771]


In [6]:
# ── Load Model ────────────────────────────────
print(f"Loading model: {MODEL_NAME}")

# num_labels=2 → binary classification
# (Positive vs Negative)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels = 2
)

# Label mappings
model.config.id2label = {0: "NEGATIVE", 1: "POSITIVE"}
model.config.label2id = {"NEGATIVE": 0, "POSITIVE": 1}

# Count parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters()
                       if p.requires_grad)

print(f"\nModel loaded!")
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size          : ~{total_params*4/1e6:.0f} MB")

# Data collator handles dynamic padding
# Pads each batch to the longest sequence IN THAT BATCH
# More efficient than padding to max_length globally!
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(f"\nData collator ready!")

Loading model: distilbert-base-uncased


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Model loaded!
Total parameters    : 66,955,010
Trainable parameters: 66,955,010
Model size          : ~268 MB

Data collator ready!


In [7]:
# ── Compute Metrics ───────────────────────────
accuracy_metric = evaluate.load("accuracy")
f1_metric       = evaluate.load("f1")

def compute_metrics(eval_pred):
    """
    Called by Trainer after each evaluation.

    eval_pred = (logits, labels)
    logits shape: (batch_size, num_labels)
    labels shape: (batch_size,)
    """
    logits, labels = eval_pred

    # Convert logits to predictions
    # argmax → index of highest logit = predicted class
    predictions = np.argmax(logits, axis=-1)

    # Compute metrics
    accuracy = accuracy_metric.compute(
        predictions = predictions,
        references  = labels
    )
    f1 = f1_metric.compute(
        predictions = predictions,
        references  = labels,
        average     = 'weighted'
    )

    return {
        'accuracy': accuracy['accuracy'],
        'f1'      : f1['f1']
    }

print("Metrics function ready!")
print("Will compute: Accuracy + F1 Score")

Metrics function ready!
Will compute: Accuracy + F1 Score


In [9]:
# ── Training Arguments ────────────────────────
training_args = TrainingArguments(

    # Where to save checkpoints
    output_dir = "./distilbert-imdb",

    # Training duration
    num_train_epochs = 3,

    # Batch sizes
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,

    # Learning rate settings
    # 2e-5 is standard for BERT fine-tuning
    learning_rate = 2e-5,

    # Warmup: gradually increase LR for first 100 steps
    # Prevents destroying pretrained weights early on!
    warmup_steps  = 100,

    # L2 regularization (prevents overfitting)
    weight_decay  = 0.01,

    # Evaluation and saving
    eval_strategy = "epoch",
    save_strategy = "epoch",  # save after each epoch
    load_best_model_at_end = True,  # keep best checkpoint!

    # Logging
    logging_steps = 50,
    logging_dir   = "./logs",

    # Reproducibility
    seed = 42,
)

# Calculate training steps
steps_per_epoch = len(tokenized_train) // \
                  training_args.per_device_train_batch_size
total_steps     = steps_per_epoch * \
                  training_args.num_train_epochs

print("Training Configuration:")
print(f"  Epochs          : {training_args.num_train_epochs}")
print(f"  Batch size      : {training_args.per_device_train_batch_size}")
print(f"  Learning rate   : {training_args.learning_rate}")
print(f"  Warmup steps    : {training_args.warmup_steps}")
print(f"  Steps per epoch : {steps_per_epoch}")
print(f"  Total steps     : {total_steps}")
print(f"  Weight decay    : {training_args.weight_decay}")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training Configuration:
  Epochs          : 3
  Batch size      : 16
  Learning rate   : 2e-05
  Warmup steps    : 100
  Steps per epoch : 187
  Total steps     : 561
  Weight decay    : 0.01


In [12]:
# ── Create Trainer ────────────────────────────
trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = tokenized_train,
    eval_dataset    = tokenized_test,
    processing_class  = tokenizer,
    data_collator   = data_collator,
    compute_metrics = compute_metrics,
)

print("Starting fine-tuning...")
print("="*55)

# Train!
train_result = trainer.train()

print("\n" + "="*55)
print("TRAINING COMPLETE!")
print("="*55)
print(f"Training loss    : {train_result.training_loss:.4f}")
print(f"Training time    : {train_result.metrics['train_runtime']:.0f}s")
print(f"Samples/second   : {train_result.metrics['train_samples_per_second']:.1f}")

Starting fine-tuning...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.360499,0.300974,0.858000,0.857881
2,0.247262,0.318691,0.870000,0.869507
3,0.151197,0.388414,0.878000,0.877749


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



TRAINING COMPLETE!
Training loss    : 0.2998
Training time    : 263s
Samples/second   : 34.2


In [13]:
# ── Evaluate ──────────────────────────────────
print("Evaluating on test set...")
results = trainer.evaluate()

print("\n" + "="*55)
print("EVALUATION RESULTS")
print("="*55)
print(f"Accuracy : {results['eval_accuracy']:.4f} "
      f"({results['eval_accuracy']*100:.2f}%)")
print(f"F1 Score : {results['eval_f1']:.4f}")
print(f"Loss     : {results['eval_loss']:.4f}")

Evaluating on test set...



EVALUATION RESULTS
Accuracy : 0.8580 (85.80%)
F1 Score : 0.8579
Loss     : 0.3010


In [14]:
# ── Test on Custom Reviews ────────────────────
print("="*55)
print("TESTING ON CUSTOM REVIEWS")
print("="*55)

from transformers import pipeline as hf_pipeline

# Create pipeline with our fine-tuned model
classifier = hf_pipeline(
    "text-classification",
    model     = model,
    tokenizer = tokenizer,
    device    = 0 if torch.cuda.is_available() else -1
)

# Custom reviews
reviews = [
    "This movie was absolutely incredible! Best film I've seen in years.",
    "Terrible waste of time. The plot made no sense whatsoever.",
    "The acting was decent but the story was a bit slow.",
    "A masterpiece of cinema! The director did an outstanding job.",
    "I fell asleep halfway through. Extremely boring and predictable.",
    "Not bad, not great. Just an average film overall.",
]

print("\nCustom Review Predictions:")
print("-"*55)
for review in reviews:
    result = classifier(review)[0]
    label  = result['label']
    score  = result['score']
    emoji  = "😊" if label == "POSITIVE" else "😞"
    print(f"{emoji} {label} ({score:.4f})")
    print(f"   '{review[:60]}...'")
    print()

TESTING ON CUSTOM REVIEWS

Custom Review Predictions:
-------------------------------------------------------
😊 POSITIVE (0.9443)
   'This movie was absolutely incredible! Best film I've seen in...'

😞 NEGATIVE (0.9304)
   'Terrible waste of time. The plot made no sense whatsoever....'

😞 NEGATIVE (0.8452)
   'The acting was decent but the story was a bit slow....'

😊 POSITIVE (0.9459)
   'A masterpiece of cinema! The director did an outstanding job...'

😞 NEGATIVE (0.9191)
   'I fell asleep halfway through. Extremely boring and predicta...'

😞 NEGATIVE (0.6946)
   'Not bad, not great. Just an average film overall....'



In [15]:
# Push to HuggingFace Hub
from huggingface_hub import login

from google.colab import userdata
login(token=userdata.get("HF_TOKEN"))

# Push model and tokenizer
model_name = "Ved2001/distilbert-imdb-sentiment"

print(f"Pushing model to: {model_name}")
trainer.push_to_hub(model_name)

print(f"\nModel live at:")
print(f"huggingface.co/{model_name}")
print("Anyone can now use your model with:")
print(f'pipeline("text-classification", model="{model_name}")')

Pushing model to: Ved2001/distilbert-imdb-sentiment


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


Model live at:
huggingface.co/Ved2001/distilbert-imdb-sentiment
Anyone can now use your model with:
pipeline("text-classification", model="Ved2001/distilbert-imdb-sentiment")
